In [ ]:
%load_ext tensorboard
!rm -rf "./logs"

import os
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from datetime import datetime
from tensorflow import keras
from tensorflow.keras import layers
from tensorboard.plugins.hparams import api as hp

import warnings
warnings.filterwarnings("ignore")

!wget "https://raw.githubusercontent.com/ibrahimerdem/application_storage/main/models.py" -P models -nc
!wget "https://raw.githubusercontent.com/ibrahimerdem/application_storage/main/aux_func.py" -P aux_func -nc
import sys
sys.path.append("models")
sys.path.append("aux_func")

from aux_func import *
from models import *

print(tf.version.VERSION)
print(keras.__version__)

gpus = tf.config.list_physical_devices("GPU")        
logical_gpus = tf.config.list_logical_devices('GPU')
print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPU")

from psutil import *

!lscpu|grep "Model name"
!nvidia-smi -L

In [ ]:
!wget "https://raw.githubusercontent.com/ibrahimerdem/application_storage/main/data_set.csv" -P data_set -nc

data = pd.read_csv(
    "data-set/data-set.csv",
     sep=";")

In [ ]:
def model_eval_timebasis(model, test_x, hold_df, c_list, k=5):
    hit = 0
    cus = 0
    mrr = 0
    retk = set()
    relk = set()
    p = 0
    r = 0
    y_pred = model.predict(test_x)
    cust = c_list
    counter = len(cust)
    for i in range(counter):
        c = cust[i]
        rel = set(hold_df.loc[hold_df["c_id"]==c, "item_code"].values)
        if len(rel) > 0:
            ret = y_pred[i][-1].argsort()[::-1][:k]
            cus += 1
            retk = retk.union(set(ret))
            inter = set.intersection(rel, ret)
            if len(inter) > 0:
                rank = 5
                for j in inter:
                    th = np.where(ret==j)
                    if th[0][0]<=rank:
                        rank = th[0][0]  
                mrr += 1/(rank+1)
                hit += 1
                p += len(inter)/len(ret)
                r += len(inter)/len(rel)

    return p/cus, r/cus, mrr/cus, hit/cus, len(retk)

def model_eval_lastevent(model, test_x, test_y, k=5):
    hit = 0
    cus = 0
    mrr = 0
    retk = set()
    relk = set()
    y_pred = model.predict(test_x)
    cust = test_df["c_id"].unique()
    counter = len(cust)
    for i in range(counter):
        c = cust[i]
        rel = test_y[i][-1]
        ret = y_pred[i][-1].argsort()[::-1][:k]
        retk = retk.union(set(ret))
        r = np.where(ret==int(rel))
        if len(r[0]) > 0:
            rr = 1/(r[0][0]+1)
            hit += 1
        else:
            rr = 0
        mrr = mrr + rr

    return hit/counter, mrr/counter, len(retk)

def event_last_k(k=5):
    hit = 0
    mrr = 0
    cus = 0
    retk = set()
    cust = test_data["c_id"].unique()
    counter = len(cust)
    for i in range(counter):
        c = cust[i]
        rel = (test_data.loc[test_data["c_id"]==c, "output_seq"].values)[0][-1]
        re = test_data.loc[test_data["c_id"]==c, "input_seq"]
        ret = np.array(list(set([int(z) for z in re.values[0][:][::-1]])))[:k]
        if len(ret)==k:
            cus += 1
            retk = retk.union(set(ret))
            r = np.where(ret==int(rel))
            if len(r[0]) > 0:
                rr = 1/(r[0][0]+1)
                hit = hit + 1
            else:
                rr = 0
            mrr = mrr + rr
    
    return hit/cus, mrr/cus, len(retk)

def time_last_k(k=5):
    cus = 0
    hit = 0
    mrr = 0
    retk = set()
    p = 0
    r = 0
    cust = test_data["c_id"].unique()
    counter = len(cust)
    for i in range(counter):
        c = cust[i]
        rel = set(hold_df.loc[hold_df["c_id"]==c, "item_code"].values)
        if len(rel) > 0:
            re = test_data.loc[test_data["c_id"]==c, "input_seq"]
            ret = np.array(list(set([int(z) for z in re.values[0][:][::-1]])))[:k]
            if len(ret)==k:
                cus += 1
                retk = retk.union(set(ret))
                inter = set.intersection(rel, ret)
                if len(inter) > 0:
                    rank = 1000
                    for j in inter:
                        th = np.where(ret==j)
                        if th[0][0]<=rank:
                            rank = th[0][0]    
                    mrr += 1/(rank+1)
                    hit += 1
                    p += len(inter)/len(ret)
                    r += len(inter)/len(rel)
    
    return p/cus, r/cus, mrr/cus, hit/cus, len(retk)

def get_inputs(hparams):
    if hparams["model"]==1:
        input_list=[hparams["train_list"][0]]
        eval_list=[hparams["test_list"][0]]
    elif hparams["model"]==2:
        input_list=[hparams["train_list"][0], hparams["train_list"][1]]
        eval_list=[hparams["test_list"][0], hparams["test_list"][1]]
    elif hparams["model"]==3:
        input_list=[hparams["train_list"][0], hparams["train_list"][1],
                    hparams["train_list"][5]]
        eval_list=[hparams["test_list"][0], hparams["test_list"][1],
                   hparams["test_list"][5]]
    elif hparams["model"]==4:
        input_list=[hparams["train_list"][0], hparams["train_list"][1],
                    hparams["train_list"][5], hparams["train_list"][6]]
        eval_list=[hparams["test_list"][0], hparams["test_list"][1],
                   hparams["test_list"][5], hparams["test_list"][6]]
    elif hparams["model"]==5:
        input_list=[hparams["train_list"][0], hparams["train_list"][1],
                    hparams["train_list"][5], hparams["train_list"][6],
                    hparams["train_list"][2]]
        eval_list=[hparams["test_list"][0], hparams["test_list"][1],
                   hparams["test_list"][5], hparams["test_list"][6],
                   hparams["test_list"][2]]
    elif hparams["model"]==6:
        input_list=[hparams["train_list"][0], hparams["train_list"][1],
                    hparams["train_list"][5], hparams["train_list"][6],
                    hparams["train_list"][4]]
        eval_list=[hparams["test_list"][0], hparams["test_list"][1],
                   hparams["test_list"][5], hparams["test_list"][6],
                   hparams["test_list"][4]]
    elif hparams["model"]==7:
        input_list=[hparams["train_list"][0], hparams["train_list"][1],
                    hparams["train_list"][5], hparams["train_list"][6],
                    hparams["train_list"][3]]
        eval_list=[hparams["test_list"][0], hparams["test_list"][1],
                   hparams["test_list"][5], hparams["test_list"][6],
                   hparams["test_list"][3]]
    else:
        input_list=[hparams["train_list"][0], hparams["train_list"][1],
                    hparams["train_list"][5], hparams["train_list"][6],
                    hparams["train_list"][2], hparams["train_list"][3]]
        eval_list=[hparams["test_list"][0], hparams["test_list"][1],
                   hparams["test_list"][5], hparams["test_list"][6],
                   hparams["test_list"][2], hparams["test_list"][3]]

    return input_list, eval_list

def run(hparams, model, time=False):
    print(f"Experiment:{hparams['model']}-{hparams['style']}-{hparams['n_padded']}-{hparams['k']}-{hparams['replication']}")
    input_list, eval_list = get_inputs(hparams)
    batch_size = 128
    val_split = 0.2
        
    hist = model_training(model,
                          input_list,
                          hparams["train_list"][-1],
                          val_split,
                          batch_size=batch_size)
    if time:
        pre, rec, mrr, hit, retk = model_eval_timebasis(hist.model,
                                                        eval_list,
                                                        hold_df,
                                                        test_data["c_id"].unique())
        return pre, rec, mrr, hit, retk
    else:
        hit, mrr,retk = model_eval_lastevent(hist.model,
                                             eval_list,
                                             hparams["test_list"][-1],
                                             hparams["k"])
        return hit, mrr, retk

In [ ]:
df = data.copy() 
pro_max = df["item_code"].astype("int64").max()
rec_max = df["recency"].astype("int64").max()
pay_max = df["payment"].astype("int64").max()
cat_max = df["category_code"].astype("int64").max()
met_max = df["method_code"].astype("int64").max()
mon_max = df["month"].astype("int64").max()
day_max = df["dayofweek"].astype("int64").max()

hold_c = df["c_id"].unique()

emb_units = [96, 64, 128, 96, 64, 128, 128, 64]
rnn_units = [100, 100, 200, 200, 200, 200, 200, 200]
drop_rates = [0.8, 0.6, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8]
n_paddeds = [24, 32]
styles = ["lstm"]

exp_df = pd.DataFrame(columns=["model", "n_padded", "k", "style",
                               "replication", "mrr", "hit", "coverage"])

for n_padded in n_paddeds:
    for n in range(0,9):
        for s in styles:
            for r in range(1,11):
                train_c, test_c = train_test_split(hold_c, test_size=0.2, shuffle=True)
                train_df = df[df["c_id"].isin(train_c)]
                test_df = df[df["c_id"].isin(test_c)]
                train_data = create_sequences(train_df)
                train_data = create_splits(train_data)
                train_list = make_padding(train_data, n_padded)
                test_data = create_sequences(test_df)
                test_data = create_splits(test_data)
                test_list = make_padding(test_data, n_padded)
                if n>0:
                    hparams = {
                        "bidirect": False,
                        "style": s,
                        "model": n,
                        "emb_unit": emb_units[n-1],
                        "rnn_unit": rnn_units[n-1],
                        "dropout": drop_rates[n-1],
                        "learning": 0.001,
                        "replication": r,
                        "n_padded": n_padded,
                        "item_max": pro_max+1,
                        "rec_max": rec_max+1,
                        "pay_max": pay_max+1,
                        "cat_max": cat_max+1,
                        "met_max": met_max+1,
                        "mon_max": mon_max+1,
                        "day_max": day_max+1,
                        "k": 5,
                        "train_list": train_list,
                        "test_list": test_list
                    }
                    model = model_base(hparams)
                    hit, mrr, retk = run(hparams, model)
                    exp_df = exp_df.append(
                                        {"model": n,
                                         "n_padded": n_padded,
                                         "k": 5,
                                         "style": s,
                                         "replication":r,
                                         "mrr": mrr,
                                         "hit": hit,
                                         "coverage": retk/pro_max}, ignore_index=True)
                else:
                    hit_last, mrr_last, retk_last = event_last_k()
                    exp_df = exp_df.append(
                                        {"model": "last-k",
                                         "n_padded": n_padded,
                                         "k": 5,
                                         "style": "last-k",
                                         "replication":r,
                                         "mrr": mrr_last,
                                         "hit": hit_last,
                                         "coverage": retk_last/pro_max}, ignore_index=True)


exp_df.to_csv("experiment_results.csv", sep=";", index=False)

In [ ]:
# df = data.copy()
# df["t_date"] = df["t_date"].astype("datetime64")

# emb_units = [96, 64, 128, 96, 64, 128, 128, 64]
# rnn_units = [100, 100, 200, 200, 200, 200, 200, 200]
# drop_rates = [0.8, 0.6, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8]
# n_paddeds = [32]
# styles = ["lstm"]

# exp_df = pd.DataFrame(columns=["model", "n_padded", "k", "style",
#                                "replication", "precision", "recall",
#                                "mrr", "hit", "coverage"])

# model_start = "07-01-2016"
# model_end = "04-01-2018"
# test_start = "04-01-2018"
# test_end = "07-01-2018"
# df_base = df[(df["t_date"]>=model_start)&(df["t_date"]<model_end)]
# hold_df = df.loc[(df["t_date"]>=test_start)&(df["t_date"]<test_end), ["c_id", "item_code"]]
# items = df_base["item_code"].unique()
# custs = df_base["c_id"].unique()
# hold_df = hold_df[hold_df["item_code"].isin(items)]
# hold_df = hold_df[hold_df["c_id"].isin(custs)]
# hold_c = hold_df["c_id"].unique()

# pro_max = df_base["item_code"].astype("int64").max()
# rec_max = df_base["recency"].astype("int64").max()
# pay_max = df_base["payment"].astype("int64").max()
# cat_max = df_base["category_code"].astype("int64").max()
# met_max = df_base["method_code"].astype("int64").max()
# mon_max = df_base["month"].astype("int64").max()
# day_max = df_base["dayofweek"].astype("int64").max()
                
# for n_padded in n_paddeds:
#     for n in range(0,8):
#         for s in styles:
#             for r in range(1,6):
#                 _, test_c = train_test_split(hold_c, test_size=0.9, shuffle=True)
#                 train = df_base[~(df_base["c_id"].isin(test_c))]
#                 test = df_base[df_base["c_id"].isin(test_c)]
#                 train_data = create_sequences(train)
#                 train_data = create_splits(train_data)
#                 train_list = make_padding(train_data, n_padded)
#                 test_data = create_sequences(test)
#                 test_data.rename(columns={"item_code":"input_seq"}, inplace=True)
#                 test_list = make_padding(test_data, n_padded)

#                 hparams = {
#                     "bidirect": False,
#                     "style": s,
#                     "model": n+1,
#                     "emb_unit": emb_units[n],
#                     "rnn_unit": rnn_units[n],
#                     "dropout": drop_rates[n],
#                     "learning": 0.001,
#                     "replication": r,
#                     "n_padded": n_padded,
#                     "item_max": pro_max+1,
#                     "rec_max": rec_max+1,
#                     "pay_max": pay_max+1,
#                     "cat_max": cat_max+1,
#                     "met_max": met_max+1,
#                     "mon_max": mon_max+1,
#                     "day_max": day_max+1,
#                     "k": 5,
#                     "train_list": train_list,
#                     "test_list": test_list
#                 }
#                 model = model_base(hparams)
#                 pre, rec, mrr, hit, retk = run(hparams, model, True)
#                 exp_df = exp_df.append(
#                                     {"model": n+1,
#                                     "n_padded": n_padded,
#                                     "k": 5,
#                                     "style": s,
#                                     "replication":r,
#                                     "precision": pre,
#                                     "recall": rec,
#                                     "mrr": mrr,
#                                     "hit": hit,
#                                     "coverage": retk/pro_max}, ignore_index=True)


# exp_df.to_csv("experiment_results.csv", sep=";", index=False)

In [ ]:
exp_df